# Monte Carlo Methods

Let's add Gymnasium to our environment

In [ ]:
!pip install gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 13.5 MB/s eta 0:00:00


## Helpers

Now let's define some helper functions.

In [ ]:
# Video management imports
import cv2
import matplotlib.pyplot as plt

# Check if we running in Google Colab or Jupyter Notebook
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab')
    # Do you need to connect with Google Drive? Uncomment the next two lines
    # from google.colab import drive
    # drive.mount('/content/drive')
    # This auxiliary function simplifies the visualization of OpenCV Images
    from google.colab.patches import cv2_imshow
else:
    print('Running in Jupyter Notebook')
    # This auxiliary function simplifies the visualization of OpenCV Images
    def cv2_imshow(img, title=''):
        if img.ndim > 2:
            img= cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img)
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()
        else:
            # view the image in its natural size
            plt.figure(figsize=(img.shape[1], img.shape[0]), dpi=1)
            plt.imshow(img, cmap='gray')
            plt.title(title)
            plt.xticks([]), plt.yticks([])
            plt.show()

# Helper functions to save videos and images
def save_video(img_array, path='/content/test.mp4'):
  height, width, layers = img_array[0].shape
  size = (width, height)
  out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MP4V'), 15, size)
  if out.isOpened():
    for i in range(len(img_array)):
      bgr_img = cv2.cvtColor(img_array[i], cv2.COLOR_RGB2BGR)
      out.write(bgr_img)
    out.release()
    print('Video saved.')
  else:
    print(f'Could not save video path: {path}')

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def to_html_video(img_array):
  # Function to update each frame in the animation
  def update_frame(i):
      frame.set_data(img_array[i])
      return frame,

  # Set up the plot
  # Your list of images
  # images = [img1, img2, img3, ...]
  # Assuming all images have the same shape as the first image
  img_height, img_width = img_array[0].shape[:2]

  # Set up the figure size based on the image size with a desired DPI
  dpi = 100
  fig_width = img_width / float(dpi)
  fig_height = img_height / float(dpi)
  fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi)
  ax.axis('off')

  frame = ax.imshow(img_array[0])

  # Create an animation
  ani = FuncAnimation(fig, update_frame, frames=len(img_array), blit=True)

  # To display the animation inline, convert it to HTML and display it
  html = HTML(ani.to_html5_video())
  plt.close(fig)
  return html

Running in Google Colab


## Let it go, the FrozenLake environment

Info [FrozenLake-v1 environment](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)


In [ ]:
import gymnasium as gym
import numpy as np
from tqdm.notebook import tqdm
import plotly.express as px # Install with: conda install -c plotly plotly_express

print(f"Gym version: {gym.__version__}")

Gym version: 1.0.0


## Monte Carlo Agent

The MC update that is implemented in the `step(episode)` method to update the Q-values is:

$$ Q_{t+1}(s, a) = Q_t(s, a) + \frac{1}{N(s, a)}(G_t - Q_t(s, a))$$

Where: $ G_t = R_{t+1} + \gamma G_{t+1} $ is the incremental form of the return, and $ \frac{1}{N(s, a)} $ is the number of times this combination of (state, action) has appeared in the episode.

First, we will define a helper function to implement argmax handling the case where there are several values with the maximum value (a tie). In this case, the function will select randomly among them. Thus, we avoid overrepresenting certain actions.

In [ ]:
# Helper function representing the argmax function. This implementation
# finds all the elements matching the maximun value and selects one,
# randomly, to break the tie.
def argmax(np_array):
    """argmax method with random tie-breaking.

    Args:
      np_array: A numpy array.
    Returns:
      Index of one of the appearances of the highest value in the array.
    """
    tie_indices = np.flatnonzero(np_array == np_array.max())
    return np.random.choice(tie_indices)

Next we define our Agent.

We will use the dictionary `visits` indexed with a tuple (state, action) to store the number of visits.

In [ ]:
class MonteCarloAgent:
    # Initialize the main variables of the agent
    def __init__(self, /, num_states, num_actions, epsilon=0.1, gamma=1.0):
        self.num_states = num_states
        self.num_actions = num_actions
        self.q_values = np.zeros((self.num_states, self.num_actions))
        self.epsilon = epsilon
        self.gamma = gamma
        self.visits = {}

    # Choose the action according to the currently learned policy, i.e. selecting
    # the action that maximized the value.
    def choose_policy_action(self, observation):
        return argmax(self.q_values[observation, :])

    # Choose the action following an epsilon-greedy policy
    def choose_e_greedy_action(self, observation):
        # We select with a given probability a random action
        if np.random.rand() < self.epsilon:
            return np.random.choice(range(self.num_actions))
        # or otherwise we follow the policy action.
        else:
            return self.choose_policy_action(observation)

    # Update the Q values, based on a list of tuples of the form
    # (observation, action, reward) containing the information of an episode
    def update_q(self, episode):
        g = 0 # Initialize the accumulated reward with 0
        # We will iterate through every step in reverse order
        for observation, action, reward in reversed(episode):
            # Update the accumulated reward
            g = reward + self.gamma * g
            # Update the visit count for the tuple (observation, action)
            self.visits[(observation, action)] = self.visits.get((observation, action), 0) + 1
            # Update the Q-value
            self.q_values[observation, action] += 1.0 / self.visits[(observation, action)] * (g - self.q_values[observation, action])

## Training

In [ ]:
EPISODES = 250_000

# Initialize the environment
env = gym.make('FrozenLake-v1', render_mode='rgb_array')

# Initialize the agent
agent = MonteCarloAgent(num_states=env.observation_space.n, num_actions=env.action_space.n, epsilon=0.1, gamma=1.0)

# We will track how many times we visit a state during training
visit_count = np.zeros(agent.num_states)

# Training loop
for episode_number in tqdm(range(EPISODES)):
    # Generate an episode
    episode = []
    done = False
    # Reset and obtain initial observation
    observation, _ = env.reset()
    # Select a random action to start
    action = np.random.choice(range(agent.num_actions))
    # We will count the number of steps in the episode
    step_count = 0

    while not done:
        # perform the action and obtain a new observation
        new_observation, reward, terminated, truncated, _ = env.step(action)
        # check if we're done
        done = terminated or truncated
        # append data (observation, action, and reward) to the episode
        episode.append((observation, action, reward))
        # replace old observation for next step
        observation = new_observation
        # select next action using epsilon-greedy
        action = agent.choose_e_greedy_action(observation)
        # update visit count for observation
        visit_count[observation] += 1
        # update step count
        step_count += 1
        # end episode if we are over 100 steps
        if step_count > 100:
            done = True

    # Update the Q values
    agent.update_q(episode)

print('Done!')

  0%|          | 0/250000 [00:00<?, ?it/s]

Done!


## Testing the Agent

### Performance

To understand the performance of our training, we will visualize how many times each state has been visited.

As we can see the exploration is not overly effective as the agent has spent most of the time exploring the states close to the starting point.

As we use complete episodes, we spend most of the time in areas well explorer, and we only explore areas closer to the target in a few cases.

In [ ]:
fig = px.imshow(visit_count.reshape((4, 4)))
fig.show()

### The Learned Policy

Finally let's see how our agent behaves, running 5 episodes:

In [ ]:
images = []
for _ in range(5):
    # reset per episode variables starting with the environment
    observation, _ = env.reset()
    count = 0
    done = False
    while not done:
        action = agent.choose_policy_action(observation)
        observation, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        image = env.render()
        images.append(image)
        count += 1
        if count > 100:
            done = True

In [ ]:
to_html_video(images)

# Appendix

## `np_array.max()`
Obtains the highest value on the array.

In [ ]:
test_array = np.array([0, 7, 3, 8, 4, 3, 9, 9, 2])
test_array.max()

9

## `np.flatnonzero(np_array)`

This method can be used to get the indices of the values in the array that are different from zero or zero-like (e.g. False).

In [ ]:
test_array = np.array([0, 7, 4, 0, 7, 5, 0, 3])
np.flatnonzero(test_array)

array([1, 2, 4, 5, 7])

In [ ]:
test_array = np.array([False, True, False, True, True, False])
np.flatnonzero(test_array)

array([1, 3, 4])

## `np.random.choice(np_array)`
Returns a random sample from the given array.

In [ ]:
test_array = np.array([4, 7, 11, 1, 8])
np.random.choice(test_array)

7